# Non-CSS surface-code decoding

This notebook decodes the full, non-CSS GKP surface-code lattice. It mirrors the CSS-sector example, but keeps the coupled `q` and `p` quadratures together.

In [1]:
using Random
using LinearAlgebra
using LatticeDecoder

Random.seed!(2);

function is_not_logical_error(logical_check, residual; atol = 1e-5)
    logical_coordinates = logical_check' * residual
    return all(abs(x - round(x)) < atol for x in logical_coordinates)
end;



Build a distance-3 surface code and derive the full parity-check matrix `H` and correction generator `G` in the `qqpp` convention.

In [2]:
d = 3

code = GKP_Surface_Code(d, false);
M = code.code;
J = code.J;

H = -M * J;
G = J * inv(M);
logical_check = inv(H);

logicals = code.logical;


Draw one full displacement vector and decode it with serial belief propagation.

In [3]:
noise_std = 0.5 / sqrt(2 * pi)
max_iter = size(H, 2)
decoder = "nearest";

error_vector = sample_error(noise_std, size(H, 2));
received = copy(error_vector);

tanner_graph = initialize_tanner_graph(H);
bp_estimate = run_serial_belief_propagation!(
    tanner_graph,
    received,
    noise_std,
    max_iter,
    decoder;
);

decoded_integer_correction = hard_decision(bp_estimate, H);


correction = received - G * decoded_integer_correction;
residual = error_vector - correction;

# println(logical_check' * residual)

println("BP correct: ", is_not_logical_error(logical_check, residual))
println("BP + X correct: ", is_not_logical_error(logical_check, residual + logicals[1, :]))
println("BP + Z correct: ", is_not_logical_error(logical_check, residual + logicals[2, :]))
println("BP + Y correct: ", is_not_logical_error(logical_check, residual + logicals[1, :] + logicals[2, :]))


any_correct = is_not_logical_error(logical_check, residual) ||
    is_not_logical_error(logical_check, residual + logicals[1, :]) ||
    is_not_logical_error(logical_check, residual + logicals[2, :]) ||
    is_not_logical_error(logical_check, residual + logicals[1, :] + logicals[2, :])

println("Any correct: ", any_correct)


BP correct: false
BP + X correct: false
BP + Z correct: false
BP + Y correct: false
Any correct: false
